# 10.2 Jacobi 旋转与 QR 特征值迭代

幂法一次只锁定少数目标特征值；本节转向全部特征值问题。对称 Jacobi 方法用正交旋转消去非对角元，QR 迭代用反复相似变换把矩阵推向近似对角形式。

In [ ]:
import pathlib
import sys

import numpy as np

ROOT = pathlib.Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next(parent for parent in pathlib.Path.cwd().parents if (parent / 'src').exists())
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    jacobi_eigenvalue_method,
    off_diagonal_frobenius_norm,
    qr_eigenvalue_iteration,
)

## 非对角范数

对称矩阵被正交对角化后，非对角元素应接近 0。因此 Jacobi 和 QR 迭代都可以用非对角部分的 Frobenius 范数作为停止准则。

In [ ]:
A = np.array([
    [4.0, 1.0, 0.0],
    [1.0, 3.0, 1.0],
    [0.0, 1.0, 2.0],
])

off_diagonal_frobenius_norm(A)

## Jacobi 旋转

Jacobi 方法每步选择一个非对角元 $a_{pq}$，构造只作用在 $(p,q)$ 平面的正交旋转。旋转后该位置被消去，同时累计旋转矩阵得到正交特征向量近似。

In [ ]:
jacobi = jacobi_eigenvalue_method(A, tolerance=1e-12)
jacobi.eigenvalues, jacobi.iterations, jacobi.off_diagonal_norm

In [ ]:
jacobi.eigenvectors.T @ jacobi.eigenvectors

In [ ]:
np.linalg.norm(A @ jacobi.eigenvectors - jacobi.eigenvectors @ np.diag(jacobi.eigenvalues))

## QR 特征值迭代

QR 迭代先做 $A_k=Q_kR_k$，再令 $A_{k+1}=R_kQ_k$。这两个矩阵相似，所以特征值不变；对称情形下，迭代通常会逐步趋近对角矩阵。

In [ ]:
B = np.array([[2.0, 1.0], [1.0, 2.0]])
qr = qr_eigenvalue_iteration(B, tolerance=1e-12)
np.sort(qr.eigenvalues), qr.iterations, qr.off_diagonal_norm

In [ ]:
np.column_stack([
    np.arange(qr.off_diagonal_history.size),
    qr.off_diagonal_history,
])[:8]

## 小结

* Jacobi 旋转保留对称性，并直接累积正交特征向量。
* QR 迭代通过相似变换保留特征值，并把对称矩阵推向近似对角形式。
* 非对角范数是全部特征值算法的自然误差监视量。